# Faruq-v3 — Cross-Model Hard-Confusion Consensus Audit

Validation-only, post-training, tanpa inference ulang dan tanpa training. Audit memakai enam event JSON seed42 (CMC0, STB1, AF2, IGEM1, SAF1, ACMC1) untuk mencari confusion family yang berulang lintas model. Test tidak dibuka.

Gate deskriptif dibekukan sebelum hasil dibaca: family disebut recurring-hard jika muncul pada minimal 4 dari 6 model dan total support minimal 8 object-errors.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib, json, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/multimodel-complementarity-audit'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
SRC = str(REPO / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())


In [ ]:
BASE = 'experiments/faruq-v3-multimodel-complementarity-seed42-v1/events'
NAMES = ['CMC0','STB1','AF2','IGEM1','SAF1','ACMC1']
REQUIRED = tuple(f'{BASE}/{name}_seed42_events.json' for name in NAMES)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
EVENTS = [require_project_artifact(PROJECT_ROOT, rel) for rel in REQUIRED]
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-cross-model-hard-confusion-consensus-seed42-v1/cross_model_hard_confusion_consensus.json'
for name, path in zip(NAMES, EVENTS):
    print(name, '->', path)
print('OUT ->', OUTPUT)


In [ ]:
command = [sys.executable, '-m', 'coffee_detector.analysis.cross_model_hard_confusion_consensus']
for path in EVENTS:
    command += ['--event', str(path)]
command += ['--output', str(OUTPUT)]
subprocess.run(command, cwd=REPO, check=True)
result = json.loads(OUTPUT.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
print('AUDIT SELESAI:', OUTPUT)


In [ ]:
import pandas as pd
from IPython.display import display

print('CONSENSUS COUNTS:', result['consensus_counts'])
print('\nMODEL ERROR SUMMARIES')
rows = []
for model, summary in result['model_summaries'].items():
    rows.append({
        'model': model,
        'classification_errors': summary['classification_errors_iou50'],
        'top3_directed_share': summary['directed_concentration']['top3_share'],
        'top5_directed_share': summary['directed_concentration']['top5_share'],
        'directed_entropy': summary['directed_concentration']['normalized_entropy'],
    })
display(pd.DataFrame(rows).style.format({
    'top3_directed_share':'{:.2%}',
    'top5_directed_share':'{:.2%}',
    'directed_entropy':'{:.3f}',
}))

for key, title in [('directed_families','DIRECTED'),('undirected_families','UNDIRECTED'),('gt_classes','GT CLASS')]:
    print('\n===', title, 'CONSENSUS RANKING ===')
    table = pd.DataFrame(result[key])
    cols = ['family','models_with_error','model_fraction','total_support','mean_support_per_model','max_single_model_support','models_present','frozen_consensus_hard_family']
    display(table[cols].head(25).style.format({
        'model_fraction':'{:.2%}',
        'mean_support_per_model':'{:.2f}',
    }))

print('Kirim CONSENSUS COUNTS + MODEL ERROR SUMMARIES + top 15 DIRECTED + top 15 UNDIRECTED. Jangan membuka test.')
